# Lesson 3: Reflection and Blogpost Writing

## Setup

In [1]:
llm_config = {"model": "gpt-3.5-turbo"}

## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [4]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])

In [5]:
print(reply)

Title: Unleashing the Power of DeepLearning.AI

Dive into the world of artificial intelligence with DeepLearning.AI! Founded by the prominent AI expert Andrew Ng, this platform offers top-notch courses to help you master deep learning. From neural networks to computer vision, DeepLearning.AI provides the tools and knowledge to excel in this cutting-edge field. Whether you're a seasoned professional or just starting, these courses are designed to accommodate all levels of expertise. Join the AI revolution and unlock your potential with DeepLearning.AI today!


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Unleashing the Power of DeepLearning.AI

Dive into the world of artificial intelligence with DeepLearning.AI! Founded by the prominent AI expert Andrew Ng, this platform offers top-notch courses to help you master deep learning. From neural networks to computer vision, DeepLearning.AI provides the tools and knowledge to excel in this cutting-edge field. Whether you're a seasoned professional or just starting, these courses are designed to accommodate all levels of expertise. Join the AI revolution and unlock your potential with DeepLearning.AI today!

--------------------------------------------------------------------------------
Critic (to Writer):

This blogpost effectively introduces the reader to DeepLearning.AI, highl

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
       DeepLearning.AI. Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Unleashing the Power of DeepLearning.AI

Dive into the world of artificial intelligence with DeepLearning.AI! Founded by the prominent AI expert Andrew Ng, this platform offers top-notch courses to help you master deep learning. From neural networks to computer vision, DeepLearning.AI provides the tools and knowledge to excel in this cutting-edge field. Whether you're a seasoned professional or just starting, these courses are designed to accommodate all levels of expertise. Join the AI revolution and unlock your potential with DeepLearning.AI today!

--------------------------------------------------------------------------------

********************************************************************************
Starting a n


--------------------------------------------------------------------------------
Critic (to Writer):

Aggregated Feedback:
- Content is well-written and informative about DeepLearning.AI.
- Suggested improvements include incorporating keywords, optimizing meta tags, and adding internal links for better SEO performance.

Final Suggestion:
The overall consensus from the reviewers is that the content is well-written and informative. To enhance its SEO performance, it is recommended to incorporate relevant keywords, optimize meta tags, and add internal links. By implementing these suggestions, the content can potentially reach a wider audience and improve its visibility online.

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Master Deep Learning with DeepLearning.AI

Embark on your artificial intelligence journey with DeepLearning.AI, the brainchild of AI luminary Andrew Ng. Explore neural networks, computer vision, and more th

## Get the summary

In [15]:
print(res.summary)

Title: Master Deep Learning with DeepLearning.AI

Embark on your artificial intelligence journey with DeepLearning.AI, the brainchild of AI luminary Andrew Ng. Explore neural networks, computer vision, and more through expertly crafted courses. Suitable for all skill levels, this platform equips you to thrive in the AI landscape. Unleash your potential and join the AI revolution today! To further enhance your experience and reach, be sure to optimize keywords, meta tags, and internal links for improved SEO performance. Deepen your understanding and broaden your horizons with DeepLearning.AI.
